In [2]:
#| hide
#| eval: false
! [ -e /content ] && pip install -Uqq fastai # upgrade fastai on colab

In [3]:
#| default_exp data.external

In [4]:
#| export
from __future__ import annotations
from fastdownload import FastDownload
from functools import lru_cache
from xcube.imports import *
import xcube.data

In [5]:
#| hide
from nbdev.showdoc import *
%load_ext autoreload
%autoreload 2

# Downloading...

> Helper functions to download XML datasets and pretrained models

This module is the xcube downloading counterpart of [fastai's External data](https://northeastern.zoom.us/j/94529633646). Specifically, [untar_data](https://docs.fast.ai/data.external.html#untar_data) is repleaced with `untar_xxx`.

To download any of the datasets or pretrained weights, simply run `untar_xxx` by passing any dataset name mentioned above like so: 

```python 
path = untar_xxx(XURLs.MIMIC3_L2R)
path.ls()

>> (#1) [Path('/home/deb/.xcube/data/mimic3/l2r')]
```

To download model pretrained weights: 
```python 
path = untar_xxx(XURLs.)
path.ls()

>> (#2) []
```

In [7]:
#| export
@lru_cache(maxsize=None)
def xcube_cfg() -> Config: # Config that contains default download paths for `data`, `model`, `storage` and `archive`
    "`Config` object for xcube's `config.ini`"
    return Config(Path(os.getenv('XCUBE_HOME', '~/.xcube')), 'config.ini', create=dict(
        data = 'data', archive = 'archive', storage = 'tmp', model = 'models'))

This is a basic `Config` file that consists of `data`, `model`, `storage` and `archive`. 
All future downloads occur at the paths defined in the config file based on the type of download. For example, all future xcube datasets are downloaded to the `data` while all pretrained model weights are download to `model` unless the default download location is updated.

In [8]:
cfg = xcube_cfg()
cfg.data, cfg.path('archive')

('data', Path('/home/deb/.xcube/archive'))

In [9]:
#| export
def xcube_path(folder:str) -> Path: 
    "Local path to `folder` in `Config`"
    return xcube_cfg().path(folder)

In [10]:
xcube_path('archive')

Path('/home/deb/.xcube/archive')

## URLs -

In [ ]:
#| export
class XURLs():
    "Global cosntants for datasets and model URLs."
    LOCAL_PATH = Path.cwd()
    S3 = 'https://xcubebucket.s3.us-east-2.amazonaws.com/'
    
    #main datasets
    MIMIC3 = f'{S3}mimic3/mimic3.tgz'
    MIMIC3_DEMO = f'{S3}mimic3/mimic3_demo.tgz'
    MIMIC3_L2R = f'{S3}mimic3/mimic3_l2r/mimic3_l2r.tgz'
    MIMIC3_L2R_DEMO = f'{S3}mimic3/mimic3_l2r/mimic3_l2r_demo.tgz'
    MIMIC4 = f'{S3}mimic4/mimic4.tgz'
    MIMIC4_DEMO = f'{S3}mimic4/mimic4_demo.tar.gz'
    MIMIC4_L2RBOOT_MISTRAL3B = f'{S3}mimic4/mimic4_l2rboot_mistral3b.tar.gz'
    MIMIC4_L2RBOOT_MISTRAL7B = f'{S3}mimic4/mimic4_l2rboot_mistral7b.tar.gz'
    MIMIC4_L2R = f'{S3}mimic4/mimic4_l2r/mimic4_l2r.tgz'
    AMAZON_670K = f'{S3}amazon-670k/Amazon-670K.tar.gz'
    WIKI10_31K = f'{S3}wiki10-31k/Wiki10-31K.tgz'
    EURLEX_4K = f'{S3}eurlex-4k/Eurlex-4k.tgz'
    LF_AMAZON_131K= f'{S3}lf-amazon-131k/LF-Amazon-131K.tar.gz'
    LF_AMAZON_131K_sample= f'{S3}lf-amazon-131k/LF-Amazon-131K_sample.tar.gz'
    LF_WIKISEEALSO_320K= f'{S3}lf-wikiseealso-320k/LF-WikiSeeAlso-320K.tar.gz'
    LF_WIKISEEALSO_320K_sample= f'{S3}lf-wikiseealso-320k/LF-WikiSeeAlso-320K_sample.tar.gz'
    LF_WIKIPEDIA_500K= f'{S3}lf-wikipedia-500k/LF-Wikipedia-500K.tar.gz'
    LF_AMAZON_1M= f'{S3}lf-amazon-1m/LF-Amazon-1M.tar.gz'
    AMAZON_3M=f'{S3}Amazon-3M/amazon-3m.tar.gz'
    
    def path(
        url:str='.', # File to download
        c_key:str='archive' # Key in `Config` where to save URL
    ) -> Path:
        "Local path where to download based on `c_key`"
        fname = url.split('/')[-1]
        local_path = XURLs.LOCAL_PATH/('models' if c_key=='model' else 'data')/fname
        if local_path.exists(): return local_path
        return xcube_path(c_key)/fname

The default local path is at `~/.xcube/archive/` but this can be updated by passing a different `c_key`. Note: `c_key` should be one of `'archive', 'data', 'model', 'storage'`.

In [ ]:
url = XURLs.MIMIC3_L2R
local_path = XURLs.path(url)
test_eq(local_path.parent, xcube_path('archive'))
local_path

Path('/home/deb/.xcube/archive/mimic3_l2r.tgz')

In [ ]:
local_path = XURLs.path(url, c_key='model')
test_eq(local_path.parent, xcube_path('model'))
local_path

Path('/home/deb/.xcube/models/mimic3_l2r.tgz')

## untar_xxx -

In [ ]:
#| export
def untar_xxx(
    url:str, # File to download
    archive:Path=None, # Optional override for `Config`'s `archive` key
    data:Path=None, # Optional override for `Config`'s `data` key
    c_key:str='data', # Key in `Config` where to extract file
    force_download:bool=False, # Setting to `True` will overwrite any existing copy of data
    base:str='~/.xcube' # Directory containing config file and base of relative paths
) -> Path: # Path to extracted file(s)
    "Download `url` using `FastDownload.get`"
    d = FastDownload(xcube_cfg(), module=xcube.data, archive=archive, data=data, base=base)
    return d.get(url, force=force_download, extract_key=c_key)

`untar_xxx` is a thin wrapper for `FastDownload.get`. It downloads and extracts `url`, by default to subdirectories of `~/.xcube`, and returns the path to the extracted data. Setting the `force_download` flag to 'True' will overwrite any existing copy of the data already present. For an explanation of the `c_key` parameter, see `XURLs`.

In [ ]:
p = untar_xxx(XURLs.MIMIC3_L2R)
p

Path('/home/deb/.xcube/data/mimic3_l2r')

In [ ]:
list(p.glob('**/*.csv'))

[]

## Export -

In [12]:
#| hide
import nbdev; nbdev.nbdev_export()